In [898]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from itertools import combinations
from collections import Counter, defaultdict
import joblib
import random
import re

font_path = "C:/Windows/Fonts/gulim.ttc"
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc("font", family=font)

In [899]:
rate_df = pd.read_csv("rate.csv").drop('Unnamed: 0',axis=1)
rate_df

,Year,Home_Eng_Nation,Away_Eng_Nation,home_rate
0,2022,아르헨티나,프랑스,0.666667
1,2022,크로아티아,모로코,0.000000
2,2022,프랑스,모로코,0.933333
3,2022,아르헨티나,크로아티아,0.333333
4,2022,모로코,포르투갈,0.000000
...,...,...,...,...
312,2002,아르헨티나,나이지리아,1.000000
313,2002,스페인,슬로베니아,1.000000
314,2002,우루과이,덴마크,0.000000
315,2002,독일,사우디아라비아,1.000000


In [900]:
team_df = pd.read_csv("Final DF.csv")
team_df

,Year,Nation,Eng_Nation,Wc_Rank,Wc_Point,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,...,FS_7,FS_8,FS_9,FS_10,FS_11,FS_12,FS_13,FS_14,ATK_INDEX,DEF_INDEX
0,2002,브라질,Brazil,1,21,0.46,2.352113,1.33,818.33,0.00,...,75.27,77.05,72.73,82.09,31.95,68.95,59.61,62.32,0.141500,-0.070093
1,2002,독일,Germany,2,16,0.55,1.817073,8.67,718.33,0.00,...,66.38,67.86,72.57,80.57,37.60,62.88,63.29,63.48,0.031444,0.011481
2,2002,터키,Turkey,3,13,0.46,1.457627,33.00,593.33,0.00,...,66.45,69.80,67.05,74.05,30.85,65.18,56.78,57.60,-0.039941,0.015588
3,2002,대한민국,South Korea,4,11,0.48,2.216667,42.00,573.00,0.00,...,65.91,65.50,65.68,72.45,32.64,57.66,62.23,57.09,0.040250,0.133021
4,2002,스페인,Spain,5,11,0.65,3.227273,5.00,742.33,0.00,...,75.50,78.23,78.09,83.27,35.05,69.80,64.55,71.05,-0.105762,-0.099683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,2022,덴마크,Denmark,28,1,0.61,2.446429,13.33,1611.83,-18.00,...,72.19,70.69,67.92,70.15,17.13,66.37,53.93,55.04,NaN,NaN
188,2022,세르비아,Serbia,29,1,0.48,1.536232,31.33,1492.66,-18.00,...,66.19,72.58,69.38,64.15,18.21,64.19,51.35,54.31,NaN,NaN
189,2022,웨일스,Wales,30,1,0.44,1.233333,21.00,1545.20,10.75,...,66.19,63.88,55.19,66.27,17.62,59.62,53.12,45.12,NaN,NaN
190,2022,캐나다,Canada,31,0,0.48,2.296875,66.33,1356.74,-34.00,...,66.48,69.76,60.36,65.60,16.50,56.82,43.71,47.96,NaN,NaN


In [901]:
len(team_df)

192

In [902]:
def make_group(df, teams_per_group=4, seed=None):
    if seed is not None:
        random.seed(seed)

    n_teams = len(df)
    n_groups = n_teams // teams_per_group
    groups = {chr(ord("A") + i): [] for i in range(n_groups)}

    # 시드별로 팀 나누기
    n_seeds = teams_per_group
    teams_per_seed = n_groups

    seeds = []
    for i in range(n_seeds):
        start = i * teams_per_seed
        end = (i + 1) * teams_per_seed
        seeds.append(df.index[start:end])

    # 각 조에 시드별로 한 팀씩 배치
    for seed_group in seeds:
        shuffled_indices = list(seed_group)  # Index → list 변환
        random.shuffle(shuffled_indices)
        for group_idx, team_idx in enumerate(shuffled_indices):
            group_name = chr(ord("A") + group_idx)
            groups[group_name].append(team_idx)

    return groups

In [903]:
groups = make_group(X, seed=42)

In [904]:
groups

{'A': [3, 11, 19, 26],
 'B': [4, 15, 21, 25],
 'C': [6, 10, 18, 24],
 'D': [7, 8, 20, 31],
 'E': [2, 12, 17, 28],
 'F': [5, 14, 22, 30],
 'G': [0, 13, 23, 29],
 'H': [1, 9, 16, 27]}

In [905]:
def get_match_result(model, home, away, isgroupstage=False, threshold=0.4):
    cols = home.loc["Q_WR":"FS_14"].index
    diff = home[cols] - away[cols]

    diff_df = pd.DataFrame(diff).T  # 1행 DataFrame

    mask = (rate_df['Year'] == home['Year']) & \
        (rate_df['Home_Eng_Nation'] == home['Nation']) & \
        (rate_df['Away_Eng_Nation'] == away['Nation'])

    home_rate_val = rate_df.loc[mask, 'home_rate'].values
    if len(home_rate_val) > 0:
        diff_df['home_rate'] = home_rate_val[0]
    else:
        diff_df['home_rate'] = 0  # 매칭값 없으면 기본값 0
    diff_df = diff_df.to_numpy(dtype=float).reshape(1, -1)
    pred = model.predict_proba(diff_df)[0]
    if isgroupstage:
        if pred[0] > threshold and pred[1] > threshold:
            return 0
        elif pred[1] > pred[0]:
            return 1
        else:
            return -1

    else:
        if pred[1] > pred[0]:
            return 1
        else:
            return -1

In [906]:
def do_groupstage(model, team_df, groups):
    result = {}
    if len(team_df) == 32:
        n_result = 16
    elif len(team_df) == 48:
        n_result = 32
        thrid_lst = {}
    for g in groups.keys():
        matches = {}
        for idx in groups[g]:
            matches[idx] = 0
        for home, away in list(combinations(groups[g], 2)):
            match_result = get_match_result(
                model, team_df.iloc[home], team_df.iloc[away], isgroupstage=True
            )
            if match_result == 1:
                matches[home] += 3
            elif match_result == -1:
                matches[away] += 3
            else:
                matches[home] += 1
                matches[away] += 1
        result[g] = sorted(matches, key=lambda x: matches[x],reverse=True)[:3]
        if n_result == 32:
            thrid_lst[result[g][2]] = [matches[result[g][2]], g]
        result[g] = result[g][:2]
    if n_result == 32:
        sorted_lst = sorted(thrid_lst, key=lambda x: thrid_lst[x][0], reverse=True)[:8]
        for i in sorted_lst:
            result[thrid_lst[i][1]].append(i)
            pass
    return result

In [907]:
def make_tornament(groups):
    lst = []
    for i in groups.values():
        for ii in i:
            lst.append(ii)
    random.shuffle(lst)
    return lst

# def make_tornament(team_df, group):
#     # 1, 2, 3위 추출
#     first_place = {k: v[0] for k, v in group.items()}
#     second_place = {k: v[1] for k, v in group.items() if len(v) > 1}
#     third_place_candidates = [(k, v[2]) for k, v in group.items() if len(v) > 2]
    
#     # 3위 배정
#     used_third = []
#     def assign_third(allowed_groups):
#         candidates = [team for team in third_place_candidates 
#                       if team[0] in allowed_groups and team[1] not in used_third]
#         if not candidates:
#             return None
#         choice = random.choice(candidates)
#         used_third.append(choice[1])
#         return choice[1]

#     # 32강 매치업
#     matches = []
#     round_32 = [
#         (first_place['E'], assign_third(["A","B","C","D","F","G","H","J","K","L"])),
#         (first_place['I'], assign_third(["A","B","C","D","F","G","H","J","K","L"])),

#         (second_place['A'], second_place['B']),
#         (first_place['F'], second_place['C']),

#         (first_place['C'], second_place['F']),
#         (second_place['E'], second_place['I']),

#         (first_place['A'], assign_third(["B","C","D","E","F","G","H","I","J","K"])),
#         (first_place['L'], assign_third(["B","C","D","E","F","G","H","I","J","K"])),

#         (second_place['K'], second_place['L']),
#         (first_place['H'], second_place['J']),

#         (first_place['D'], assign_third(["A","B","C","E","F","H","I","J","K","L"])),
#         (first_place['G'], assign_third(["A","B","C","E","F","H","I","J","K","L"])),

#         (first_place['J'], second_place['H']),
#         (second_place['D'], second_place['G']),

#         (first_place['B'], assign_third(["A","C","D","E","F","G","H","I","J","L"])),
#         (first_place['K'], assign_third(["A","C","D","E","F","G","H","I","J","L"])),
#     ] 


#     for i, match in enumerate(round_32):
#         matches.append(match)

#     flat_list = [x for t in matches for x in t]
#     print(flat_list)

In [908]:
def do_tornament(model, team_df, groups):
    if len(X) == 32:
        rounds = ["16강", "8강", "4강", "준우승"]
    else:
        rounds = ["32강", "16강", "8강", "4강", "준우승"]
    tornament = make_tornament(groups)
    final={'조별리그':[]}
    for i in range(len(team_df)):
        if i not in tornament:
            final['조별리그'].append(i)
    for round in rounds:
        final[round] = []
        result_matches = []
        for i in range(0, len(tornament), 2):
            r = get_match_result(
                model, team_df.iloc[tornament[i]], team_df.iloc[tornament[i + 1]]
            )
            if r == 1:
                result_matches.append(tornament[i])
                final[round].append(tornament[i + 1])
            elif r == -1:
                result_matches.append(tornament[i + 1])
                final[round].append(tornament[i])
        tornament = result_matches
    final["우승"] = tornament
    return final

In [909]:
def do_WorldCup(model, X):
    groups = make_group(X)
    result_group = do_groupstage(model, X, groups)
    final = do_tornament(model, X, result_group)
    return final

In [910]:
def do_simulate(model, X, iter=100):
    if len(X) == 32:
        rounds = ["조별리그","16강", "8강", "4강", "준우승", "우승"]
    else:
        rounds = ["조별리그","32강", "16강", "8강", "4강", "준우승", "우승"]
    final = {}
    for round in rounds:
        final[round] = []
    for i in range(iter):
        result = do_WorldCup(model, X)
        for r in result.keys():
            final[r] = final[r] + result[r]
    return final

In [911]:
def analyze_simulation(sim_results, df):
    # 전체 라운드
    rounds = list(sim_results.keys())

    # 팀별 라운드 도달 횟수 저장
    team_stats = defaultdict(lambda: Counter())
    for rnd, results in sim_results.items():
        for team_idx in results:
            team_stats[team_idx][rnd] += 1

    # DataFrame 생성
    data = []
    for team_idx, counts in team_stats.items():
        total_plays = sum(counts.values())  # 팀이 시뮬레이션에서 등장한 총 횟수
        row = {
            "Nation": df.loc[team_idx, "Nation"] if team_idx in df.index else team_idx
        }
        for rnd in rounds:
            row[rnd] = counts[rnd] / total_plays if total_plays > 0 else 0
        data.append(row)

    result_df = pd.DataFrame(data).fillna(0)

    return result_df


def normalize_win_prob(result_df):
    total_win_prob = result_df["우승"].sum()
    scaled_df = result_df[["Nation", "우승"]].copy()
    if total_win_prob > 0:
        scaled_df["우승"] = scaled_df["우승"] / total_win_prob
    return scaled_df.sort_values(by="우승", ascending=False).reset_index(drop=True)

In [912]:
X = team_df[160:192].reset_index().drop('index',axis=1)
# seed_plz = ['러시아','독일','브라질','포르투갈','아르헨티나','벨기에','폴란드','프랑스','스페인','페루','스위스','잉글랜드','콜롬비아','멕시코','우루과이','크로아티아','덴마크','아이슬란드','코스타리카','스웨덴','튀니지','이집트','세네갈','이란','세르비아','나이지리아','호주','일본','모로코','파나마','대한민국','사우디아라비아']
seed_plz=['카타르','브라질','벨기에','프랑스','아르헨티나','잉글랜드','스페인','포르투갈','멕시코','네덜란드','덴마크','독일','우루과이','스위스','미국','크로아티아','세네갈','이란','일본','모로코','세르비아','폴란드','대한민국','튀니지','카메룬','캐나다','에콰도르','사우디아라비아','가나','호주','코스타리카','웨일스']
X = X.set_index('Nation').loc[seed_plz].reset_index()

In [913]:
model = joblib.load("xgb_d1_v4.pkl")
result_lst = do_simulate(model, X, 300)
result_df = analyze_simulation(result_lst, X)
scaled_win_df = normalize_win_prob(result_df)

In [914]:
from collections import Counter
count_dict = Counter(result_lst['우승'])
print(count_dict)
nation_dict={}
for i in count_dict.keys():
    nation_dict[X['Nation'].iloc[i]]=count_dict[i]
print(nation_dict)

Counter({1: 300})
{'브라질': 300}


In [919]:
result_df

,Nation,조별리그,16강,8강,4강,준우승,우승
0,카타르,1.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,미국,0.566667,0.423333,0.010000,0.000000,0.000000,0.0
2,이란,0.883333,0.116667,0.000000,0.000000,0.000000,0.0
3,일본,0.910000,0.090000,0.000000,0.000000,0.000000,0.0
4,모로코,0.770000,0.226667,0.003333,0.000000,0.000000,0.0
5,세르비아,0.546667,0.366667,0.083333,0.003333,0.000000,0.0
6,폴란드,0.783333,0.210000,0.006667,0.000000,0.000000,0.0
7,대한민국,0.896667,0.103333,0.000000,0.000000,0.000000,0.0
8,튀니지,0.923333,0.076667,0.000000,0.000000,0.000000,0.0
9,캐나다,1.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [916]:
scaled_win_df.head()

,Nation,우승
0,브라질,1.0
1,카타르,0.0
2,이란,0.0
3,미국,0.0
4,모로코,0.0


In [917]:
home = team_df.iloc[1]
away = team_df.iloc[2]
cols = home.loc["Q_WR":"FS_13"].index  # home 기준으로 컬럼 선택
diff = home[cols] - away[cols]
diff.index = [f"{c}" for c in cols]
diff

Q_WR              0.09
Q_GR          0.359446
F_Rank          -24.33
F_Point          125.0
F_Rd               0.0
F_Pd               0.0
Avg_Apps       0.21739
Avg_Age            0.0
Avg_Famous           9
FS_0              6.87
FS_1               1.2
FS_2             -9.56
FS_3              7.27
FS_4             -0.72
FS_5              -8.0
FS_6              0.64
FS_7             -0.07
FS_8             -1.94
FS_9              5.52
FS_10             6.52
FS_11             6.75
FS_12             -2.3
FS_13             6.51
dtype: object

In [918]:
result_lst

{'조별리그': [0,
  14,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  16,
  17,
  18,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  14,
  16,
  17,
  19,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  14,
  17,
  18,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  16,
  17,
  18,
  19,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  16,
  17,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  14,
  16,
  17,
  18,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  0,
  13,
  14,
  17,
  18,
  19,
  21,
  22,
  23,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  14,
  16,
  17,
  18,
  19,
  20,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  0,
  16,
  17,
  18,
  19,
  20,
  21,
 